In [ ]:
from cgi import parse
from social_groups.analysis.definitions import defs
import polars as pl
from social_groups.orchestrator.services import ExperimentConfiguration


frame: pl.DataFrame = defs().load_asset_value("combined_data")

In [ ]:
combined_data = frame
import social_groups.polars_columns as plc
from social_groups.analysis.polars_transformations.deserialize_experiment_configuration import (
    deserialize_experiment_configuration,
)

%load_ext autoreload
%autoreload 2

shape: (5, 20)
┌─────┬────────┬─────────────┬──────────────┬───┬───────┬──────────────┬──────────────┬────────────┐
│ id  ┆ run_id ┆ question_id ┆ phoenix_span ┆ … ┆ top_p ┆ model_family ┆ model_parame ┆ sizes      │
│ --- ┆ ---    ┆ ---         ┆ _url         ┆   ┆ ---   ┆ ---          ┆ ters         ┆ ---        │
│ i64 ┆ i64    ┆ i64         ┆ ---          ┆   ┆ f32   ┆ str          ┆ ---          ┆ list[str]  │
│     ┆        ┆             ┆ str          ┆   ┆       ┆              ┆ list[f64]    ┆            │
╞═════╪════════╪═════════════╪══════════════╪═══╪═══════╪══════════════╪══════════════╪════════════╡
│ 0   ┆ 0      ┆ 0           ┆ http://local ┆ … ┆ 0.7   ┆ Qwen3        ┆ [14.0, 4.0]  ┆ ["H", "M"] │
│     ┆        ┆             ┆ host:46001/p ┆   ┆       ┆              ┆              ┆            │
│     ┆        ┆             ┆ roject…      ┆   ┆       ┆              ┆              ┆            │
│ 1   ┆ 0      ┆ 1           ┆ http://local ┆ … ┆ 0.7   ┆ Qwen3        ┆ [14.0, 4.0]  ┆ ["H", "M"] │
│     ┆        ┆             ┆ host:46001/p ┆   ┆       ┆              ┆              ┆            │
│     ┆        ┆             ┆ roject…      ┆   ┆       ┆              ┆              ┆            │
│ 2   ┆ 0      ┆ 2           ┆ http://local ┆ … ┆ 0.7   ┆ Qwen3        ┆ [14.0, 4.0]  ┆ ["H", "M"] │
│     ┆        ┆             ┆ host:46001/p ┆   ┆       ┆              ┆              ┆            │
│     ┆        ┆             ┆ roject…      ┆   ┆       ┆              ┆              ┆            │
│ 3   ┆ 0      ┆ 3           ┆ http://local ┆ … ┆ 0.7   ┆ Qwen3        ┆ [14.0, 4.0]  ┆ ["H", "M"] │
│     ┆        ┆             ┆ host:46001/p ┆   ┆       ┆              ┆              ┆            │
│     ┆        ┆             ┆ roject…      ┆   ┆       ┆              ┆              ┆            │
│ 4   ┆ 0      ┆ 4           ┆ http://local ┆ … ┆ 0.7   ┆ Qwen3        ┆ [14.0, 4.0]  ┆ ["H", "M"] │
│     ┆        ┆             ┆ host:46001/p ┆   ┆       ┆              ┆              ┆            │
│     ┆        ┆             ┆ roject…      ┆   ┆       ┆              ┆              ┆            │
└─────┴────────┴─────────────┴──────────────┴───┴───────┴──────────────┴──────────────┴────────────┘

In [ ]:
from social_groups.trialrunner.utils.hydra_config import ExperimentConfig
import dagster as dg
import polars as pl
from dagster import AssetCheckSpec

import social_groups.polars_columns as plc
from social_groups.analysis.asset_checks import (
    check_correct_data_length,
    check_unique_data_connector,
)
from social_groups.analysis.polars_transformations import make_group_constellation
from social_groups.analysis.polars_transformations.deserialize_experiment_configuration import (
    deserialize_experiment_configuration,
)
from social_groups.analysis.polars_transformations.make_group_constellation import (
    make_model_family,
    parse_parameters,
)

final_changed_prompt_mad = (
    combined_data.filter(
        (pl.col("name").is_in({"changed_prompt_mad2", "changed_prompt_mad3"}))
        & (pl.col("data_connector") == "mmlu-pro-medium-subset")
    )
    .with_columns(
        deserialize_experiment_configuration(
            pl.col("experiment_configuration_json")
        ).alias("_experiment_configuration")
    )
    .with_columns(
        pl.col("_experiment_configuration")
        .struct.field("strategy")
        .struct.field("configuration")
        .struct.field("debate_agents")
        .list.eval(pl.element().struct.field("backend").struct.field("model_name"))
        .alias(plc.model_names),
    )
    .with_columns(
        make_group_constellation(),
        make_model_family(),
    )
    .drop("_experiment_configuration")
)

print(check_unique_data_connector(final_changed_prompt_mad))
print(check_correct_data_length(final_changed_prompt_mad, 300 * 3 * (27 + 9)))

print(dg.Output(final_changed_prompt_mad))

In [ ]:
final_changed_prompt_mad.head()